# 🚀 SERVE YOUR TRAINED MODEL FROM GOOGLE DRIVE
Run this in Colab to serve checkpoint-10 via ngrok

In [ ]:
# Mount Google Drive and locate checkpoint
from google.colab import drive
import os
drive.mount('/content/drive')

# Find available checkpoints
base_path = '/content/drive/MyDrive/TEKNOFEST_2025/20250815_022938/checkpoints'
print(f'Looking for checkpoints in: {base_path}')

# List available checkpoints
if os.path.exists(base_path):
    checkpoints = [d for d in os.listdir(base_path) if d.startswith('checkpoint-')]
    print(f'Available checkpoints: {checkpoints}')
    
    # Use checkpoint-10 (confirmed from your screenshot)
    CHECKPOINT_PATH = os.path.join(base_path, 'checkpoint-10')
    
    if os.path.exists(CHECKPOINT_PATH):
        print(f'✅ Found checkpoint: {CHECKPOINT_PATH}')
        # Verify essential files exist
        required_files = ['config.json', 'pytorch_model.bin', 'tokenizer.json']
        missing = [f for f in required_files if not os.path.exists(os.path.join(CHECKPOINT_PATH, f))]
        if missing:
            print(f'⚠️ Missing files: {missing}')
        else:
            print('✅ All required files present')
    else:
        print(f'❌ Checkpoint not found: {CHECKPOINT_PATH}')
        print('Available directories:')
        print(os.listdir(base_path) if os.path.exists(base_path) else 'Base path not found')
else:
    print(f'❌ Base path not found: {base_path}')
    print('Checking parent directory...')
    parent = '/content/drive/MyDrive/TEKNOFEST_2025'
    if os.path.exists(parent):
        print(f'Contents of {parent}:')
        print(os.listdir(parent))
    
print(f'Using checkpoint: {CHECKPOINT_PATH}')

In [ ]:
# Fix import issues and install requirements
!pip install --upgrade --no-cache-dir --no-deps unsloth_zoo -q
!pip install --upgrade --force-reinstall --no-deps torchvision -q  
!pip install flask flask-cors pyngrok -q

# Restart runtime to clear import conflicts
import os
os.kill(os.getpid(), 9)

In [ ]:
# Load model with fixed imports (run after runtime restart)
import unsloth  # Import first before everything else
from unsloth import FastLanguageModel
import torch
import os

print('Loading checkpoint-10...')

try:
    # Method 1: Load base model + checkpoint as adapter
    print("Loading base model first...")
    
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name="unsloth/gemma-3n-E4B-it",  # Base model
        max_seq_length=2048,
        dtype=None,  # Auto detection
        load_in_4bit=True,
        device_map="auto"
    )
    
    print("✅ Base model loaded")
    
    # Try to load your checkpoint as LoRA adapter
    if os.path.exists(CHECKPOINT_PATH):
        try:
            from peft import PeftModel
            print(f"Attempting to load LoRA from {CHECKPOINT_PATH}...")
            
            # Check if it's a LoRA checkpoint
            adapter_config = os.path.join(CHECKPOINT_PATH, 'adapter_config.json')
            if os.path.exists(adapter_config):
                model = PeftModel.from_pretrained(model, CHECKPOINT_PATH)
                print("✅ Your trained LoRA adapter loaded successfully!")
            else:
                print("⚠️ No LoRA adapter found, using base model")
                print(f"Files in checkpoint: {os.listdir(CHECKPOINT_PATH)}")
                
        except Exception as adapter_error:
            print(f"⚠️ Could not load adapter: {adapter_error}")
            print("Using base model - it will still work with general Gemma 3N capabilities")
    else:
        print(f"❌ Checkpoint path not found: {CHECKPOINT_PATH}")
        print("Using base model only")
    
    # Enable inference mode
    FastLanguageModel.for_inference(model)
    
    # Set padding token
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    
    print(f'✅ Model ready for inference')
    print(f'Model size: {sum(p.numel() for p in model.parameters())/1e9:.2f}B parameters')
    print(f'Device: {next(model.parameters()).device}')
    
except Exception as e:
    print(f'❌ Loading failed: {e}')
    print("This might be due to Colab environment issues.")
    print("Try restarting runtime and running cells again.")
    raise e

In [ ]:
# Create Flask server with SYSTEM PROMPT
from flask import Flask, request, jsonify
from flask_cors import CORS
import re

app = Flask(__name__)
CORS(app)

# SYSTEM PROMPT - This defines the AI's behavior
SYSTEM_PROMPT = """Sen gelişmiş bir Türk telekom müşteri hizmetleri asistanısın. 
Müşterilerin duygularına uygun yanıt ver ve gerekli araçları [araç_adı] formatında kullan.

Kullanabileceğin araçlar:
[get_current_balance] - Bakiye sorgula
[check_payment_history] - Ödeme geçmişi
[view_current_plan] - Mevcut paket
[check_data_usage] - İnternet kullanımı
[activate_esim] - eSIM aktivasyonu
[troubleshoot_connection] - Bağlantı sorunu
[create_support_ticket] - Destek talebi
[escalate_to_supervisor] - Yöneticiye aktar

Duygulara göre yanıt ver:
- angry: Özür dile, hızlı çözüm sun
- sad: Empati göster
- confused: Basit açıklama yap
- happy: Samimi ol
- neutral: Profesyonel ol"""

@app.route('/health', methods=['GET'])
def health():
    return jsonify({
        'status': 'healthy',
        'model': 'gemma3n-teknofest-checkpoint-10',
        'checkpoint': 'checkpoint-10',
        'device': str(torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'),
        'ready': True
    })

@app.route('/predict', methods=['POST'])
def predict():
    try:
        data = request.json
        text = data.get('text', '')
        emotion = data.get('emotion', 'neutral')
        
        # BUILD COMPLETE PROMPT WITH SYSTEM + USER + EMOTION
        prompt = f'''{SYSTEM_PROMPT}

<start_of_turn>user
<emotion>{emotion}</emotion>
{text}
<end_of_turn>
<start_of_turn>assistant'''
        
        # Tokenize
        inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
        
        # Generate response
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=200,
                temperature=0.7,
                do_sample=True,
                top_p=0.95,
                pad_token_id=tokenizer.eos_token_id
            )
        
        # Decode response
        full_response = tokenizer.decode(outputs[0], skip_special_tokens=True)
        
        # Extract assistant response
        if 'assistant' in full_response:
            response = full_response.split('assistant')[-1].strip()
        else:
            response = full_response
        
        # Extract tools from response (format: [tool_name])
        tools_found = re.findall(r'\[([^\]]+)\]', response)
        
        # Log the interaction
        print(f"Emotion: {emotion}")
        print(f"User: {text}")
        print(f"AI: {response}")
        print(f"Tools: {tools_found}")
        
        return jsonify({
            'generated_text': response,
            'emotion_detected': emotion,
            'tools_extracted': tools_found,
            'model': 'checkpoint-10',
            'success': True
        })
        
    except Exception as e:
        return jsonify({'error': str(e), 'success': False}), 500

@app.route('/model_info', methods=['GET'])
def model_info():
    return jsonify({
        'checkpoint': 'checkpoint-10',
        'path': CHECKPOINT_PATH,
        'parameters': sum(p.numel() for p in model.parameters()),
        'device': str(model.device),
        'dtype': str(model.dtype),
        'system_prompt': SYSTEM_PROMPT[:100] + '...'  # Show first 100 chars
    })

print('✅ Flask server created with SYSTEM PROMPT')

In [ ]:
# Start ngrok tunnel
from pyngrok import ngrok
import nest_asyncio
nest_asyncio.apply()

# Configure ngrok token
!ngrok config add-authtoken 31Iyz0YNz20h4XPZAhCdzH9mQfa_7pS9XT3qX1N6YC3kS6tZY

# Kill any existing tunnels
ngrok.kill()

# Start new tunnel
public_url = ngrok.connect(5000)

print('\n' + '='*60)
print('🎉 MODEL DEPLOYED SUCCESSFULLY!')
print('='*60)
print(f'Your model is accessible at: {public_url}')
print('='*60)
print('\n📋 COPY THIS URL FOR YOUR LOCAL SYSTEM:')
print(f'\n{public_url}\n')
print('='*60)
print('\nTo use with your local system:')
print(f'python3 ENTERPRISE_TELCO_PLATFORM.py {public_url}')
print('\n⚠️ Keep this cell running!')

In [ ]:
# Run the server (THIS WILL BLOCK - Keep running!)
print('🚀 Starting model server...')
print('DO NOT STOP THIS CELL!')
app.run(port=5000, debug=False)

## 🧪 Test Your Model (Optional)
Run this in a separate cell to test

In [ ]:
# Test the model directly
test_prompt = '''<start_of_turn>user
<emotion>angry</emotion>
Faturamı öğrenmek istiyorum!
<end_of_turn>
<start_of_turn>assistant'''

inputs = tokenizer(test_prompt, return_tensors='pt').to(model.device)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=150,
        temperature=0.7,
        do_sample=True,
        top_p=0.95
    )

response = tokenizer.decode(outputs[0], skip_special_tokens=True)
print('Model response:')
print(response.split('assistant')[-1].strip())